In [33]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
import talib
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
import torch.optim as optim
import os
from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
import seaborn as sns

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau 
import shap
import plotly.graph_objs as go
import plotly.offline as pyo
from tqdm.auto import tqdm
from sklearn.utils.class_weight import compute_class_weight



In [34]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("gpu")
else:
    device = torch.device('cpu')
print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
print('cuDNN version:', torch.backends.cudnn.version())

gpu
2.1.2+cu121
CUDA available: True
CUDA version: 12.1
cuDNN version: 8902


In [35]:
start_date = '2018-01-01'
end_date = '2024-01-24'

stock_data = yf.download("AAPL", start=start_date, end=end_date)[["Adj Close", "High", "Low", "Volume"]]

stock_data = stock_data.reset_index()

stock_data = stock_data[['Date', 'Adj Close', "High", "Low", "Volume"]]

stock_data = stock_data.sort_values(by="Date")
stock_data.head(45)

[*********************100%%**********************]  1 of 1 completed


,Date,Adj Close,High,Low,Volume
0,2018-01-02,40.722878,43.075001,42.314999,102223600
1,2018-01-03,40.715790,43.637501,42.990002,118071600
2,2018-01-04,40.904911,43.367500,43.020000,89738400
3,2018-01-05,41.370628,43.842499,43.262501,94640000
4,2018-01-08,41.216953,43.902500,43.482498,82271200
5,2018-01-09,41.212234,43.764999,43.352501,86336000
6,2018-01-10,41.202774,43.575001,43.250000,95839600
7,2018-01-11,41.436813,43.872501,43.622501,74670800
8,2018-01-12,41.864700,44.340000,43.912498,101672400
9,2018-01-16,41.651932,44.847500,44.035000,118263600


In [36]:
time_step = 44

In [37]:
test_index = int((len(stock_data)-44)*0.8+44+44)

In [38]:
date = stock_data["Date"].iloc[time_step:].dt.strftime('%Y-%m-%d')
date_test = stock_data["Date"].iloc[test_index:].reset_index()
date_test.drop(columns=["index"], inplace=True)
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [39]:
def add_technical_indicators(data, timeperiod=time_step):

    # MACD
    macd, macdsignal, macdhist = talib.MACD(data["Adj Close"], fastperiod=12, slowperiod=26, signalperiod=9)

    rsi = talib.RSI(data["Adj Close"], timeperiod=14)

    # CMO
    cmo = talib.CMO(data["Adj Close"], timeperiod=timeperiod)

    # MOM
    mom = talib.MOM(data["Adj Close"], timeperiod=timeperiod)

    # Bollinger Bands
    upperband, middleband, lowerband = talib.BBANDS(data["Adj Close"], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)

    # SMA
    sma_s = talib.SMA(data["Adj Close"], timeperiod=20)
    sma_l = talib.SMA(data["Adj Close"], timeperiod=50)

    # Calculate Exponential Moving Average (EMA)
    ema = talib.EMA(data["Adj Close"], timeperiod=timeperiod)

    # Calculate Stochastic Oscillator
    slowk, slowd = talib.STOCH(data['High'], data['Low'], data['Adj Close'], fastk_period=14, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)

    # Calculate Average True Range (ATR)
    atr = talib.ATR(data['High'], data['Low'], data['Adj Close'], timeperiod=14)

    # Calculate On-Balance Volume (OBV)
    obv = talib.OBV(data['Adj Close'], data['Volume'])

    # Combine all indicators into a DataFrame
    indicators = pd.DataFrame({
        'MACD': macd,
        'MACD_signal': macdsignal,
        'RSI': rsi,
        'CMO': cmo,
        'MOM': mom,
        'Upper_BB': upperband,
        'Middle_BB': middleband,
        'Lower_BB': lowerband,
        'SMA_SHORT': sma_s,
        'SMA_LONG': sma_l,
        'EMA': ema,
        'SLOWK': slowk,
        'SLOWD': slowd,
        'ATR': atr,
        'OBV': obv,

    })
    return indicators

In [40]:
indicators = add_technical_indicators(stock_data)
indicators.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0


In [41]:
indicators_with_price = pd.concat([indicators, stock_data["Adj Close"]], axis=1, join='inner')
indicators_with_price.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0,40.722878
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0,40.715790
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0,40.904911
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0,41.370628
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0,41.216953
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0,41.212234
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0,41.202774
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0,41.436813
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0,41.864700
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0,41.651932


In [42]:
indicators_with_price = indicators_with_price.dropna()
indicators_with_price = indicators_with_price.reset_index(drop=True)
indicators_with_price

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.567014,0.443929,58.241527,7.100792,1.143600,43.382888,41.728834,40.074780,41.728834,40.814299,41.035721,-7.013330,-3.909606,2.715222,-3.124152e+08,42.355835
1,0.548494,0.464842,58.592094,7.332800,1.202904,43.261031,41.862708,40.464386,41.862708,40.847955,41.096608,-20.016706,-8.448319,2.714432,-2.214400e+08,42.405678
2,0.515805,0.475035,57.044809,6.516152,0.819328,43.280287,41.922406,40.564525,41.922406,40.878762,41.148143,-27.992068,-18.340701,2.690138,-3.790588e+08,42.256142
3,0.432812,0.466590,50.806424,3.052123,-0.254189,43.245417,41.956469,40.667521,41.956469,40.892874,41.168692,-36.985407,-28.331394,2.648797,-5.128460e+08,41.610512
4,0.361721,0.445616,50.674720,2.976500,-0.055668,43.183914,41.996703,40.809492,41.996703,40.897387,41.187696,-46.752088,-37.243188,2.644561,-5.914436e+08,41.596264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,188.796999,189.292038,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993
1471,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,188.434000,189.536287,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005
1472,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,188.164999,189.787603,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998
1473,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,188.117999,190.033787,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999


In [43]:
# Irrelelvant
indicators_with_price['Prev_Adj_Close'] = indicators_with_price['Adj Close'].shift(1)
indicators_with_price['Return'] = ((indicators_with_price['Adj Close'] - indicators_with_price['Prev_Adj_Close'])/indicators_with_price['Prev_Adj_Close'])*100
indicators_with_price['Signal'] = np.where(indicators_with_price['Return'] > 1, 1,
                                           np.where(indicators_with_price['Return'] < -1, 2, 0))
indicators_with_price


,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Prev_Adj_Close,Return,Signal
0,0.567014,0.443929,58.241527,7.100792,1.143600,43.382888,41.728834,40.074780,41.728834,40.814299,41.035721,-7.013330,-3.909606,2.715222,-3.124152e+08,42.355835,NaN,NaN,0
1,0.548494,0.464842,58.592094,7.332800,1.202904,43.261031,41.862708,40.464386,41.862708,40.847955,41.096608,-20.016706,-8.448319,2.714432,-2.214400e+08,42.405678,42.355835,0.117676,0
2,0.515805,0.475035,57.044809,6.516152,0.819328,43.280287,41.922406,40.564525,41.922406,40.878762,41.148143,-27.992068,-18.340701,2.690138,-3.790588e+08,42.256142,42.405678,-0.352632,0
3,0.432812,0.466590,50.806424,3.052123,-0.254189,43.245417,41.956469,40.667521,41.956469,40.892874,41.168692,-36.985407,-28.331394,2.648797,-5.128460e+08,41.610512,42.256142,-1.527896,2
4,0.361721,0.445616,50.674720,2.976500,-0.055668,43.183914,41.996703,40.809492,41.996703,40.897387,41.187696,-46.752088,-37.243188,2.644561,-5.914436e+08,41.596264,41.610512,-0.034241,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,188.796999,189.292038,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993,183.630005,-0.517351,0
1471,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,188.434000,189.536287,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005,182.679993,3.257068,1
1472,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,188.164999,189.787603,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998,188.630005,1.553301,1
1473,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,188.117999,190.033787,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999,191.559998,1.216330,1


In [44]:
# Not important
indicators_with_price["Signal"].value_counts()

Signal
0    737
1    414
2    324
Name: count, dtype: int64

In [45]:
indicators_with_price.dropna(inplace=True)
indicators_with_price

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Prev_Adj_Close,Return,Signal
1,0.548494,0.464842,58.592094,7.332800,1.202904,43.261031,41.862708,40.464386,41.862708,40.847955,41.096608,-20.016706,-8.448319,2.714432,-2.214400e+08,42.405678,42.355835,0.117676,0
2,0.515805,0.475035,57.044809,6.516152,0.819328,43.280287,41.922406,40.564525,41.922406,40.878762,41.148143,-27.992068,-18.340701,2.690138,-3.790588e+08,42.256142,42.405678,-0.352632,0
3,0.432812,0.466590,50.806424,3.052123,-0.254189,43.245417,41.956469,40.667521,41.956469,40.892874,41.168692,-36.985407,-28.331394,2.648797,-5.128460e+08,41.610512,42.256142,-1.527896,2
4,0.361721,0.445616,50.674720,2.976500,-0.055668,43.183914,41.996703,40.809492,41.996703,40.897387,41.187696,-46.752088,-37.243188,2.644561,-5.914436e+08,41.596264,41.610512,-0.034241,0
5,0.226727,0.401838,42.776469,-1.895769,-1.685963,43.175299,41.999076,40.822854,41.999076,40.886126,41.163972,-59.960169,-47.899222,2.611109,-7.396632e+08,40.653915,41.596264,-2.265464,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,188.796999,189.292038,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993,183.630005,-0.517351,0
1471,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,188.434000,189.536287,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005,182.679993,3.257068,1
1472,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,188.164999,189.787603,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998,188.630005,1.553301,1
1473,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,188.117999,190.033787,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999,191.559998,1.216330,1


In [46]:
indicators_with_price.columns

Index(['MACD', 'MACD_signal', 'RSI', 'CMO', 'MOM', 'Upper_BB', 'Middle_BB',
       'Lower_BB', 'SMA_SHORT', 'SMA_LONG', 'EMA', 'SLOWK', 'SLOWD', 'ATR',
       'OBV', 'Adj Close', 'Prev_Adj_Close', 'Return', 'Signal'],
      dtype='object')

In [47]:
# indicators_with_price = indicators_with_price.drop(columns=['Next_Adj_Close', 'Return'])
# indicators_with_price

In [48]:
indicators_with_price.drop(columns=['Prev_Adj_Close', "Signal"], inplace=True)
indicators_with_price.head(50)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Return
1,0.548494,0.464842,58.592094,7.332800,1.202904,43.261031,41.862708,40.464386,41.862708,40.847955,41.096608,-20.016706,-8.448319,2.714432,-2.214400e+08,42.405678,0.117676
2,0.515805,0.475035,57.044809,6.516152,0.819328,43.280287,41.922406,40.564525,41.922406,40.878762,41.148143,-27.992068,-18.340701,2.690138,-3.790588e+08,42.256142,-0.352632
3,0.432812,0.466590,50.806424,3.052123,-0.254189,43.245417,41.956469,40.667521,41.956469,40.892874,41.168692,-36.985407,-28.331394,2.648797,-5.128460e+08,41.610512,-1.527896
4,0.361721,0.445616,50.674720,2.976500,-0.055668,43.183914,41.996703,40.809492,41.996703,40.897387,41.187696,-46.752088,-37.243188,2.644561,-5.914436e+08,41.596264,-0.034241
5,0.226727,0.401838,42.776469,-1.895769,-1.685963,43.175299,41.999076,40.822854,41.999076,40.886126,41.163972,-59.960169,-47.899222,2.611109,-7.396632e+08,40.653915,-2.265464
6,0.072555,0.335982,38.805928,-4.708064,-2.298222,43.330935,41.955757,40.580579,41.955757,40.863471,41.115772,-60.364782,-55.692347,2.604321,-9.056264e+08,40.079483,-1.412982
7,-0.123099,0.244166,33.410024,-9.019895,-3.037197,43.669844,41.830426,39.991009,41.830426,40.822443,41.028466,-57.037889,-59.120947,2.589764,-1.069742e+09,39.151379,-2.315660
8,-0.126722,0.169988,48.771937,0.230770,-0.833458,43.603899,41.756843,39.909787,41.756843,40.813906,41.027644,-35.113303,-50.838658,2.699325,-9.195768e+08,41.009968,4.747187
9,-0.212000,0.093590,42.761391,-4.462404,-1.894451,43.620650,41.637566,39.654482,41.637566,40.775781,40.980123,-25.755924,-39.302372,2.704911,-1.083267e+09,39.958431,-2.564100
10,-0.311617,0.012549,40.504319,-6.346442,-1.669304,43.661175,41.499418,39.337661,41.499418,40.733080,40.915092,-23.129894,-27.999707,2.693601,-1.249941e+09,39.516922,-1.104921


In [49]:
y = indicators_with_price["Return"]
y_2 = indicators_with_price["SMA_SHORT"]
y_3 = indicators_with_price["EMA"]
y_4 = indicators_with_price["Upper_BB"]
y_5 = indicators_with_price["Middle_BB"]
y_6 = indicators_with_price["Lower_BB"]
X = np.array(date)

trace = go.Scatter(x=X, y=y, mode="lines", name="Adj Close")
trace_2 = go.Scatter(x=X, y=y_2, mode="lines", name="SMA")
trace_3 = go.Scatter(x=X, y=y_3, mode="lines", name="EMA")
trace_4 = go.Scatter(x=X, y=y_4, mode="lines", name="Upper_BB")
trace_5 = go.Scatter(x=X, y=y_5, mode="lines", name="Middle_BB")
trace_6 = go.Scatter(x=X, y=y_6, mode="lines", name="Lower_BB")



layout = go.Layout(
    title='Stock Price and Volume',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Adj Close', side='left', rangemode='tozero'),
    yaxis2=dict(title='SMA', side='right', overlaying='y', rangemode='tozero'),
    yaxis3=dict(title='EMA', side='right', overlaying='y', rangemode='tozero'),
    yaxis4=dict(title='Upper_BB', side='right', overlaying='y', rangemode='tozero'),
    yaxis5=dict(title='Middle_BB', side='right', overlaying='y', rangemode='tozero'),
    yaxis6=dict(title='Lower_BB', side='right', overlaying='y', rangemode='tozero'),
    height=600,
)

fig = go.Figure(data=[trace, trace_2, trace_3, trace_4, trace_5, trace_6], layout=layout)

# Show plot
pyo.iplot(fig)

In [50]:
# Custom Dataset
class RollingWindowDataset(Dataset):
    def __init__(self, X, y, window_size, device="gpu"):
        self.X = X.clone().detach().to(torch.float).to(device)
        self.y = y.clone().detach().to(torch.float).to(device)
        self.window_size = window_size

    def __len__(self):
        # Adjust the length to account for window size
        return len(self.X) - self.window_size 

    def __getitem__(self, idx):
        # Ensure idx is within the valid range
        if idx + self.window_size > len(self.X):
            raise IndexError("Index out of bounds")

        X_window = self.X[idx : idx + self.window_size]
        
        y_target = self.y[idx + self.window_size]  

        return X_window.clone().detach().to(torch.float), y_target.clone().detach().to(torch.float)


In [51]:
X = indicators_with_price.iloc[:,:-1]
y = indicators_with_price.iloc[:,-2]

signal_true = indicators_with_price.iloc[:,-1]
y

1        42.405678
2        42.256142
3        41.610512
4        41.596264
5        40.653915
           ...    
1470    182.679993
1471    188.630005
1472    191.559998
1473    193.889999
1474    195.179993
Name: Adj Close, Length: 1474, dtype: float64

In [52]:
train_signal_true = signal_true.iloc[:int(len(X)*0.8)]
test_signal_true = signal_true.iloc[int(len(X)*0.8):]
test_signal_true

1180    1.297136
1181    0.378186
1182   -2.168029
1183    1.466111
1184    0.592644
          ...   
1470   -0.517351
1471    3.257068
1472    1.553301
1473    1.216330
1474    0.665322
Name: Return, Length: 295, dtype: float64

In [53]:
correlation_matrix = X.corr()

# Perform hierarchical clustering to find the order of features
linked = sch.linkage(sch.distance.pdist(correlation_matrix), method='ward')
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix
correlation_matrix_ordered = correlation_matrix.iloc[cluster_order, cluster_order]

fig = go.Figure(data=go.Heatmap(
                    z=correlation_matrix_ordered,
                    x=correlation_matrix_ordered.columns,
                    y=correlation_matrix_ordered.columns,
                    colorscale='Viridis',
                    text=correlation_matrix_ordered.round(2).values,  
                    texttemplate="%{text}",
                    textfont={"size":9}  
                    ))

# Update the layout
fig.update_layout(
    title='Ordered Correlation Matrix',
    xaxis_title="Variables",
    yaxis_title="Variables",
    xaxis=dict(side='bottom'),
    yaxis=dict(autorange='reversed'),
    width=1000,  
    height=1000,  
)

# Show the figure
pyo.iplot(fig)

In [54]:
X= X.iloc[:, cluster_order]

In [55]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

y_test.head(44)

1180    149.882233
1181    150.449066
1182    147.187286
1183    149.345215
1184    150.230301
1185    147.286728
1186    143.418365
1187    140.385315
1188    147.207169
1189    147.485626
1190    146.988403
1191    145.814987
1192    142.115631
1193    140.156586
1194    141.857086
1195    141.369812
1196    143.686859
1197    144.661423
1198    142.413971
1199    135.741272
1200    133.762329
1201    131.634232
1202    131.564636
1203    134.697128
1204    131.494995
1205    131.127045
1206    129.307251
1207    125.339417
1208    128.889572
1209    129.207794
1210    124.374802
1211    125.657639
1212    124.325081
1213    128.899521
1214    129.426575
1215    130.003342
1216    132.748016
1217    132.668457
1218    134.010925
1219    135.184372
1220    134.458450
1221    134.518112
1222    137.103668
1223    140.325653
Name: Adj Close, dtype: float64

In [56]:
X_train_df = pd.DataFrame()
X_test_df = pd.DataFrame()
scaler_dict = {}

X_train_df = X_train
X_test_df = X_test

for column in X_train.columns:

    if column not in ["Adj Close", "Return"]:
        scaler = MinMaxScaler()

        X_train_scaled = scaler.fit_transform(X_train[[column]].values)
        X_train_df[column] = X_train_scaled
            
        X_test_scaled = scaler.transform(X_test[[column]].values)
        X_test_df[column] = X_test_scaled

        scaler_dict[column] = scaler


X_train_df.head(24)

features = X_train_df.columns
X_train_df

,ATR,OBV,Adj Close,Lower_BB,Upper_BB,Middle_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,MACD,MACD_signal,MOM,RSI,CMO
1,0.177936,0.261686,42.405678,0.048691,0.032213,0.037797,0.037797,0.016755,0.014072,0.483866,0.507174,0.508804,0.493383,0.498961,0.567548,0.468235
2,0.172548,0.234481,42.256142,0.049438,0.032346,0.038231,0.038231,0.016988,0.014472,0.447234,0.458578,0.506461,0.494198,0.494287,0.542886,0.458079
3,0.163379,0.211390,41.610512,0.050206,0.032106,0.038478,0.038478,0.017095,0.014632,0.405927,0.409499,0.500513,0.493523,0.481206,0.443456,0.415000
4,0.162440,0.197825,41.596264,0.051265,0.031682,0.038770,0.038770,0.017129,0.014780,0.361068,0.365720,0.495417,0.491846,0.483625,0.441356,0.414059
5,0.155021,0.172243,40.653915,0.051365,0.031623,0.038787,0.038787,0.017044,0.014595,0.300402,0.313373,0.485741,0.488346,0.463760,0.315470,0.353466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,0.850805,0.753803,146.053635,0.738088,0.800862,0.778358,0.778358,0.818604,0.829435,0.680158,0.631198,0.340107,0.343542,0.360230,0.463732,0.349553
1176,0.849089,0.770024,148.867889,0.744000,0.803314,0.782527,0.782527,0.817400,0.830427,0.747328,0.665569,0.388138,0.347930,0.321213,0.506322,0.377028
1177,0.803297,0.757360,147.455780,0.746416,0.805266,0.784731,0.784731,0.816305,0.830887,0.835637,0.739664,0.418622,0.358240,0.420018,0.480797,0.363378
1178,0.812803,0.772871,149.206024,0.747900,0.808367,0.787087,0.787087,0.815668,0.831932,0.856512,0.802536,0.453092,0.374178,0.423562,0.508857,0.380640


In [57]:
scaler_adj = MinMaxScaler()
scaler_adj.fit(X_train[["Adj Close"]].values)

X_train_df['Adj Close'] = scaler_adj.transform(X_train[['Adj Close']].values).flatten()
X_test_df['Adj Close'] = scaler_adj.transform(X_test[['Adj Close']].values).flatten()

y_train = scaler_adj.transform(y_train.values.reshape(-1,1)).flatten()
y_test = scaler_adj.transform(y_test.values.reshape(-1,1)).flatten()



X_test_df

,ATR,OBV,Adj Close,Lower_BB,Upper_BB,Middle_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,MACD,MACD_signal,MOM,RSI,CMO
1180,0.764804,0.775662,0.793797,0.751875,0.813419,0.791684,0.791684,0.814103,0.833706,0.876576,0.853920,0.499847,0.411144,0.487570,0.517394,0.387564
1181,0.724522,0.788577,0.797684,0.752021,0.816205,0.793223,0.793223,0.813227,0.835055,0.895160,0.867699,0.523637,0.432238,0.448749,0.527281,0.393265
1182,0.685712,0.778442,0.775317,0.751879,0.815520,0.792793,0.792793,0.810946,0.835217,0.907286,0.887937,0.523008,0.448973,0.379730,0.456307,0.359880
1183,0.661729,0.787383,0.790114,0.752318,0.813808,0.792104,0.792104,0.810434,0.836117,0.913913,0.901248,0.534244,0.464868,0.444493,0.497611,0.382099
1184,0.623617,0.797445,0.796184,0.752139,0.815320,0.792815,0.792815,0.809834,0.837282,0.923475,0.911342,0.547370,0.480511,0.467133,0.514479,0.391192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,0.263755,0.912203,1.018693,1.078159,1.105702,1.104494,1.104494,1.140327,1.152418,0.696649,0.699736,0.331987,0.360467,0.438975,0.219194,0.330144
1471,0.316961,0.925666,1.059493,1.079371,1.099583,1.101858,1.101858,1.142176,1.152792,0.728273,0.697238,0.358007,0.354748,0.530972,0.456177,0.434231
1472,0.316624,0.937531,1.079584,1.082407,1.093074,1.099905,1.099905,1.144078,1.154161,0.814261,0.731128,0.396416,0.358741,0.534505,0.541662,0.480565
1473,0.323439,0.947909,1.095561,1.083019,1.091863,1.099564,1.099564,1.145941,1.156274,0.926300,0.813001,0.440664,0.371807,0.555951,0.601114,0.515680


In [58]:
correlation_matrix = X_train_df.corr()

# Create the heatmap with text
fig = go.Figure(data=go.Heatmap(
                    z=correlation_matrix,
                    x=correlation_matrix.columns,
                    y=correlation_matrix.columns,
                    colorscale='Viridis',
                    text=correlation_matrix.round(2).values,  # Rounded values for display
                    texttemplate="%{text}",
                    textfont={"size":9}  # Adjust text size if necessary
                    ))

# Update the layout
fig.update_layout(
    title='Correlation Matrix',
    xaxis_title="Variables",
    yaxis_title="Variables",
    xaxis=dict(side='bottom'),
    yaxis=dict(autorange='reversed'),
    width=1000,  # or any width you desire
    height=1000,  # or any height you desire
)

# Show the figure
pyo.iplot(fig)

In [59]:
X_train_tensor = torch.tensor(X_train_df.to_numpy(), dtype=torch.float, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float, device=device)

X_test_tensor = torch.tensor(X_test_df.to_numpy(), dtype=torch.float, device=device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float, device=device)

train_data = RollingWindowDataset(X_train_tensor, y_train_tensor, window_size=time_step, device=device)
test_data = RollingWindowDataset(X_test_tensor, y_test_tensor, window_size=time_step, device=device)

print(test_data.__getitem__(0)[1])
print(test_data.__getitem__(1)[0])


tensor(0.7379, device='cuda:0')
tensor([[0.7245, 0.7886, 0.7977, 0.7520, 0.8162, 0.7932, 0.7932, 0.8132, 0.8351,
         0.8952, 0.8677, 0.5236, 0.4322, 0.4487, 0.5273, 0.3933],
        [0.6857, 0.7784, 0.7753, 0.7519, 0.8155, 0.7928, 0.7928, 0.8109, 0.8352,
         0.9073, 0.8879, 0.5230, 0.4490, 0.3797, 0.4563, 0.3599],
        [0.6617, 0.7874, 0.7901, 0.7523, 0.8138, 0.7921, 0.7921, 0.8104, 0.8361,
         0.9139, 0.9012, 0.5342, 0.4649, 0.4445, 0.4976, 0.3821],
        [0.6236, 0.7974, 0.7962, 0.7521, 0.8153, 0.7928, 0.7928, 0.8098, 0.8373,
         0.9235, 0.9113, 0.5474, 0.4805, 0.4671, 0.5145, 0.3912],
        [0.5981, 0.7914, 0.7760, 0.7537, 0.8163, 0.7941, 0.7941, 0.8092, 0.8374,
         0.9237, 0.9172, 0.5399, 0.4914, 0.4592, 0.4471, 0.3605],
        [0.5869, 0.7794, 0.7495, 0.7546, 0.8078, 0.7900, 0.7900, 0.8083, 0.8361,
         0.8756, 0.9035, 0.5112, 0.4936, 0.4080, 0.3677, 0.3216],
        [0.5854, 0.7650, 0.7287, 0.7521, 0.8019, 0.7857, 0.7857, 0.8063, 0.8339,
     

# Vanilla LSTM

<img src="/home/arda/Turkcell/RNN/LSTM/images/Screenshot from 2024-01-29 15-02-08.png" alt="Alt text">

- the output of an LSTM at a particular point in time is dependant on three things: ▹ The current long-term memory of the network — known as the cell state ▹ The output at the previous point in time — known as the previous hidden state ▹ The input data at the current time step

- LSTMs use a series of 'gates' which control how the information in a sequence of data comes into, is stored in and leaves the network.

- the forget gate decides which pieces of the long-term memory should now be forgotten (have less weight) given the previous hidden state and the new data point in the sequence.

- The next step involves the new memory network and the input gate. The goal of this step is to determine what new information should be added to the networks long-term memory (cell state), given the previous hidden state and new input data.

- The new memory network is a tanh activated neural network which has learned how to combine the previous hidden state and new input data to generate a 'new memory update vector'.This vector tells us how much to update each component of the long-term memory (cell state) of the network given the new data.

- Note that we use a tanh here because its values lie in [-1,1] and so can be negative. The possibility of negative values here is necessary if we wish to reduce the impact of a component in the cell state.

- in part 1 above, where we generate the new memory vector, there is a big problem, it doesn't actually check if the new input data is even worth remembering. This is where the input gate comes in. The input gate is a sigmoid activated network which acts as a filter, identifying which components of the 'new memory vector' are worth retaining.

- the output gate, deciding the new hidden state. To decide this, we will use three things; the newly updated cell state, the previous hidden state and the new input data.

Note: The number of LSTM units in a layer corresponds to the dimensionality of the hidden state vector at each time step. If a layer has N LSTM units, the hidden state vector at each time step will be N-dimensional.

hidden state and current input multiply and sum sigmoid function formula

In [60]:
class VanillaLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob):
        super(VanillaLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = self.hidden_size, num_layers=self.layer_size,
                            dropout=(dropout_prob if self.layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(self.hidden_size, output_dim)

    def forward(self, x):
        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

In [62]:
class ModelActioner:
    
    def __init__(self, train_data, test_data, device):
        self.train_data = train_data
        self.test_data = test_data
        self.device = device
        self.model = None
        self.optimizer = None
        self.criterion = nn.MSELoss()

    
    def train_validate(self, config, trial):

        batch_size = config["batch_size"]
        epochs = config["epochs"]
        hidden_size = config["hidden_size"]
        num_layers = 1
        learning_rate = config["learning_rate"]
        dropout_prob = config["dropout_prob"]
        weight_decay = config["weight_decay"]
        lr_step_size = config["lr_step_size"]
        gamma = config["gamma"]

        self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=1).to(self.device)

        n_splits = 5
        tscv = TimeSeriesSplit(n_splits=n_splits)

        val_losses = []

        for fold, (train_ids, val_ids) in enumerate(tscv.split(self.train_data)):
            print(f'Starting fold {fold+1}:')

            self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

            scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True) 

            train_subset = Subset(self.train_data, train_ids)
            val_subset = Subset(self.train_data, val_ids)
            
            # Creating data loader
            train_loader = DataLoader(dataset=train_subset, batch_size=batch_size, shuffle=False)
            val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False)

            # Training Loop
            for epoch in range(epochs):
                print('epochs {}/{}'.format(epoch+1,epochs))

                running_loss = 0.0
                total_sample_train = 0

                self.model.train()

                for batch_idx, (data, target) in enumerate(train_loader):
                    data, target = data.to(self.device), target.to(self.device)
                    target = target.view(-1,1)


                    self.optimizer.zero_grad()
                    preds = self.model(data)

                    loss = self.criterion(preds, target)
                    loss.backward()
                    self.optimizer.step() # Update model params

                    running_loss += loss.item() * data.size(0)
                    total_sample_train += data.size(0)

                train_loss = running_loss/total_sample_train
                #print(f"train loss: {train_loss}")

                self.model.eval()
                val_running_loss = 0.0
                total_sample_val = 0

                with torch.no_grad():

                    for batch_idx, (data, target) in enumerate(val_loader):
                        data, target = data.to(self.device), target.to(self.device)
                        target = target.view(-1,1)

                        preds = self.model(data)
                        loss = self.criterion(preds, target)

                        val_running_loss += loss.item() * data.size(0)
                        total_sample_val += data.size(0)
                
                if trial.should_prune():
                    raise optuna.TrialPruned()
                
                val_loss = val_running_loss/total_sample_val
                val_losses.append(val_loss)
                scheduler.step(val_loss)
                
                unique_step = fold * epochs + epoch
                trial.report(val_loss, unique_step)

                current_lr = self.optimizer.param_groups[0]['lr']

                print(f'Current Learning Rate: {current_lr}')
                print(f"train_loss: {train_loss}, val_loss: {val_loss}")
                
        mean_val_loss = np.mean(val_losses)
        print(f"Mean validation loss: {mean_val_loss}")
        return mean_val_loss
    
                    
    def train(self, config):

        batch_size = config["batch_size"]
        epochs = config["epochs"]
        hidden_size = config["hidden_size"]
        num_layers = 1
        learning_rate = config["learning_rate"]
        dropout_prob = config["dropout_prob"]
        weight_decay = config["weight_decay"]
        lr_step_size = config["lr_step_size"]
        gamma = config["gamma"]

        self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=1).to(self.device)

        # Update optimizer with updated lr
        self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

        # Creating data loader
        train_loader = DataLoader(dataset=self.train_data, batch_size=batch_size, shuffle=False)

        scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True)  

        # Training Loop
        for epoch in range(epochs):
            print('epochs {}/{}'.format(epoch+1,epochs))

            running_loss = 0.0
            total_sample_train = 0

            self.model.train()

            for batch_idx, (data, target) in enumerate(train_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.view(-1,1)  

                self.optimizer.zero_grad()
                preds = self.model(data)

                loss = self.criterion(preds, target)
                #loss = loss.mean()
                loss.backward()
                self.optimizer.step() # Update model params

                running_loss += loss.item() * data.size(0)
                total_sample_train += data.size(0)

            train_loss = running_loss/total_sample_train
            #print(f"train loss: {train_loss}")
            scheduler.step(train_loss)
            current_lr = self.optimizer.param_groups[0]['lr']

            print(f'Current Learning Rate: {current_lr}')
            print(f"train_loss: {train_loss}")
        
        return self.model
            
    
    def test(self, config):
        batch_size = config["batch_size"]
        all_preds = []

        test_loader = DataLoader(dataset=self.test_data, batch_size=batch_size, shuffle=False)

        running_loss = .0
        total_sample = 0

        self.model.eval()

        with torch.no_grad():

            for batch_idx, (data, target) in enumerate(test_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.view(-1,1)
                
                preds = self.model(data)
                loss = self.criterion(preds, target)

                running_loss += loss.item() * data.size(0)
                total_sample += data.size(0)

                all_preds.extend(preds.cpu().numpy())

            test_loss = running_loss/total_sample
            print(f"test_loss: {test_loss}")

        return all_preds
    


In [63]:

def objective(trial):
    config = {
        "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
        "epochs": trial.suggest_int("epochs", 50, 150),
        "hidden_size": trial.suggest_int("hidden_size", 10, 100),
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
        "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.15),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "lr_step_size": trial.suggest_int("lr_step_size", 10, 100), 
        "gamma": trial.suggest_float("gamma", 0.1, 0.9)
    }

    trainer = ModelActioner(train_data, test_data, device)

    val_loss = trainer.train_validate(config, trial)

    return val_loss

In [64]:
study_name = "Vanilla-LSTM-Tunner"
storage_url = "sqlite:///db.sqlite3"

optuna.delete_study(study_name=study_name, storage= storage_url)

study = optuna.create_study(direction='minimize', 
                            storage=storage_url, 
                            sampler=TPESampler(),
                            pruner=optuna.pruners.HyperbandPruner(
                            min_resource=30,  
                            max_resource=150,  
                            reduction_factor=3,  
                            ),
                            study_name=study_name,
                            load_if_exists=False)

pbar = tqdm(total=10, desc='Optimizing', unit='trial')

def callback(study, trial):
    # Update the progress bar
    pbar.update(1)
    pbar.set_postfix_str(f"Best Value: {study.best_value:.4f}")


study.optimize(objective, n_trials=10, callbacks=[callback])
pbar.close()

# Best hyperparameters
print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print('Value:', trial.value)
print('Params:')
for key, value in trial.params.items():
    print(f'{key}: {value}')

[I 2024-01-30 15:13:10,258] A new study created in RDB with name: Vanilla-LSTM-Tunner


Optimizing:   0%|          | 0/10 [00:00<?, ?trial/s]

Starting fold 1:
epochs 1/120
Current Learning Rate: 0.0012710346502279545
train_loss: 0.027789322610356305, val_loss: 0.0039938873803568265
epochs 2/120
Current Learning Rate: 0.0012710346502279545
train_loss: 0.00206311280980069, val_loss: 0.004354071892088368
epochs 3/120
Current Learning Rate: 0.0012710346502279545
train_loss: 0.0040147566837013555, val_loss: 0.003044894775720658
epochs 4/120
Current Learning Rate: 0.0012710346502279545
train_loss: 0.0021432883130680573, val_loss: 0.0005932987394136568
epochs 5/120
Current Learning Rate: 0.0012710346502279545
train_loss: 0.0004934209519935967, val_loss: 0.0020371779675346616
epochs 6/120
Current Learning Rate: 0.0012710346502279545
train_loss: 0.0014425492592731882, val_loss: 0.003328247929593085
epochs 7/120
Current Learning Rate: 0.0012710346502279545
train_loss: 0.002062427105173763, val_loss: 0.0028885440323772847
epochs 8/120
Current Learning Rate: 0.0012710346502279545
train_loss: 0.0015203989643946681, val_loss: 0.0016035687

[W 2024-01-30 15:13:40,423] Trial 0 failed with parameters: {'batch_size': 85, 'epochs': 120, 'hidden_size': 100, 'learning_rate': 0.0012710346502279545, 'dropout_prob': 0.004947180695087927, 'weight_decay': 0.00011223925718658074, 'lr_step_size': 46, 'gamma': 0.49757043272191026} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/arda/anaconda3/lib/python3.11/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_55296/3393379696.py", line 15, in objective
    val_loss = trainer.train_validate(config, trial)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_55296/2043730651.py", line 82, in train_validate
    preds = self.model(data)
            ^^^^^^^^^^^^^^^^
  File "/home/arda/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*a

KeyboardInterrupt: 

In [ ]:
trainer = ModelActioner(train_data, test_data, device)
model = trainer.train(trial.params)

epochs 1/79


Current Learning Rate: 0.00022078935295972353
train_loss: 0.42712533657771373
epochs 2/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.3398248373805689
epochs 3/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.23089723345548702
epochs 4/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.07332626369531525
epochs 5/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.07636017390744276
epochs 6/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.05740143414668837
epochs 7/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.03519123312709484
epochs 8/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.022575530587871956
epochs 9/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.012931673994312465
epochs 10/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.0075616575059550715
epochs 11/79
Current Learning Rate: 0.00022078935295972353
train_loss: 0.007493208540759423
epochs 12/79
Cur

In [ ]:
preds = trainer.test(trial.params)
preds = np.array(preds)

y_true = y_test[time_step:]
y_true = scaler_adj.inverse_transform(y_true.reshape(-1, 1)).flatten()
preds = scaler_adj.inverse_transform(preds.reshape(-1, 1)).flatten()


mse = mean_squared_error(y_true, preds)
print(f'Mean Squared Error: {mse:.4f}')

rmse = mean_squared_error(y_true, preds, squared=False)
print(f'Root Mean Squared Error: {rmse:.4f}')

r2 = r2_score(y_true, preds)
print(f'R² Score: {r2:.4f}')

mape = mean_absolute_percentage_error(y_true, preds)*100
print(f'mape Score: % {mape:.4f}')

test_loss: 0.0018684213240797896
Mean Squared Error: 39.7373
Root Mean Squared Error: 6.3038
R² Score: 0.8153
mape Score: % 3.0146


In [ ]:
stock_price = stock_data["Adj Close"].iloc[test_index:]
stock_price=stock_price.reset_index()
stock_price=stock_price.drop(columns=["index"])
stock_price

,Adj Close
0,140.325653
1,141.737747
2,141.071472
3,143.159805
4,145.118851
...,...
247,182.679993
248,188.630005
249,191.559998
250,193.889999


In [ ]:
signals = pd.DataFrame(preds, columns=['pred'])
signals["next_day"] = pd.DataFrame(y_true)
signals["today"] = stock_price
signals["Signal_Pred"] = (signals["today"] < signals["pred"]).astype(int)
signals["Signal_True"] = (signals["today"] < signals["next_day"]).astype(int)

signals.head(50)

,pred,next_day,today,Signal_Pred,Signal_True
0,140.881531,141.737747,140.325653,1,1
1,142.025452,141.071472,141.737747,1,0
2,143.215378,143.159805,141.071472,1,1
3,144.477158,145.118851,143.159805,1,1
4,145.809738,142.205154,145.118851,1,0
5,147.055481,143.487961,142.205154,1,1
6,148.211044,144.621628,143.487961,1,1
7,149.317261,149.981674,144.621628,1,1
8,150.518585,153.641220,149.981674,1,1
9,151.864792,150.886612,153.641220,0,0


In [ ]:
signals["Signal_Pred"].value_counts()

Signal_Pred
0    127
1    124
Name: count, dtype: int64

In [ ]:
signals["Date"] = date_test
signals

,pred,next_day,today,Signal_Pred,Signal_True,Date
0,140.881531,141.737747,140.325653,1,1,2023-01-23
1,142.025452,141.071472,141.737747,1,0,2023-01-24
2,143.215378,143.159805,141.071472,1,1,2023-01-25
3,144.477158,145.118851,143.159805,1,1,2023-01-26
4,145.809738,142.205154,145.118851,1,0,2023-01-27
...,...,...,...,...,...,...
246,180.227432,182.679993,183.630005,0,0,2024-01-16
247,179.823166,188.630005,182.679993,0,1,2024-01-17
248,179.591934,191.559998,188.630005,0,1,2024-01-18
249,179.598038,193.889999,191.559998,0,1,2024-01-19


In [ ]:
stock_price = stock_data["Adj Close"].iloc[test_index:]
stock_price=stock_price.reset_index()
stock_price=stock_price.drop(columns=["index"])
stock_price

,Adj Close
0,140.325653
1,141.737747
2,141.071472
3,143.159805
4,145.118851
...,...
247,182.679993
248,188.630005
249,191.559998
250,193.889999


In [ ]:
date_test["Date"] = date_test["Date"].dt.strftime('%Y-%m-%d')
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(date_test).flatten(), y=stock_data["Adj Close"].iloc[test_index:], mode='lines', name='TSLA Stock Price'))

# Buy and sell signals
buy_signals = signals[signals['Signal_Pred'] == 1]
sell_signals = signals[signals['Signal_Pred'] == 0]

# Ensure that buy and sell prices are aligned with the signals by matching on the 'Date' column
buy_prices = stock_data[stock_data['Date'].isin(buy_signals['Date'])]["Adj Close"]
sell_prices = stock_data[stock_data['Date'].isin(sell_signals['Date'])]["Adj Close"]


# Plot buy signals
fig.add_trace(go.Scatter(
    x=buy_signals['Date'], 
    y=buy_prices, 
    mode='markers', 
    name='Buy', 
    marker=dict(color='green', size=10, symbol='triangle-up')
))


# Plot sell signals
fig.add_trace(go.Scatter(
    x=sell_signals['Date'], 
    y=sell_prices, 
    mode='markers', 
    name='Sell', 
    marker=dict(color='red', size=10, symbol='triangle-down')
))


# Update layout
fig.update_layout(
    title='Stock Price with Buy and Sell Signals',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    height = 700,
    width=1280
)

# Show the plot
pyo.iplot(fig)

/home/arda/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [ ]:
trace_pred = go.Scatter(
    x=np.array(date_test).flatten(),
    y=signals['pred'],
    mode='lines+markers',
    name='Predicted'
)

trace_true = go.Scatter(
    x=np.array(date_test).flatten(),
    y=signals['next_day'],
    mode='lines+markers',
    name='Actual Next Day'
)

# Define the layout
layout = go.Layout(
    title='Predicted vs Actual Next Day Values',
    xaxis=dict(title='Index'),
    yaxis=dict(title='Value'),
    height=700
)

# Create the figure and add traces
fig = go.Figure(data=[trace_pred, trace_true], layout=layout)

# Show the figure
fig.show()